In [29]:
#PARAMETERS

import numpy as np 
import pandas as pd

S0 =100
K=100
r=0.06
sigma =0.25
T=1
N=3


##Finding u and d using the CRR parameters 


dt =T/N
u =np.exp(sigma*np.sqrt(dt))
d=1/u  #d= np.exp(-sigma*np.sqrt(dt))
p= (np.exp(r*dt)-d)/(u-d)
disc = np.exp(-r*dt)


print(f"dt   = {dt:.6f}")
print(f"u    = {u:.4f}")
print(f"d    = {d:.4f}")
print(f"p    = {p:.4f}")
print(f"disc = {disc:.4f}")
print(f"u*d  = {u*d:.4f}")




dt   = 0.333333
u    = 1.1553
d    = 0.8656
p    = 0.5337
disc = 0.9802
u*d  = 1.0000


In [30]:
stock = []
np.set_printoptions(legacy='1.25')
for n in range(0,N + 1):
    row = []
    for j in range(0,n + 1):
        S = S0*(u**(n-j))*(d**j)
        row.append(round(S, 4))
    stock.append(row)

for n, row in enumerate(stock):
    print(f"t={n}: {row}")

t=0: [100.0]
t=1: [115.5274, 86.5596]
t=2: [133.4658, 100.0, 74.9256]
t=3: [154.1896, 115.5274, 86.5596, 64.8552]


Finding the terminal payoffs and backward induction of the terminal payoffs to find the value of the option at all the nodes and time intervals using the risk neutral probabilities.

In [31]:
euro = []

# terminal payoffs
terminal = []

for j in range(N + 1):
    S = S0*(u**(N - j))*(d**j)
    terminal.append(max(K - S, 0))

euro.append(terminal)

# backward induction
values = terminal.copy()

for n in range(N - 1, -1, -1):

    new_values = []

    for j in range(n + 1):
        V = disc*(p*values[j]+(1 - p)*values[j + 1])
        new_values.append(V)

    euro.append(new_values)
    values = new_values

for level, vals in enumerate(reversed(euro)):
    print(f"t={level}: {[round(v,4) for v in vals]}")

t=0: [7.762]
t=1: [2.8077, 13.769]
t=2: [0.0, 6.143, 23.0943]
t=3: [0, 0, 13.4404, 35.1448]


In [32]:
american= terminal.copy()

decision_tables=[]

for n in range(N-1,-1,-1):
    new_vals=[]
    level_info=[]

    for j in range(n+1):
        S=S0*(u**(n-j))*(d**j)

        continuation =disc*(p*american[j+1]+(1-p)*american[j])
        excercise=max(K-S,0)
        value = max(continuation,excercise)
        decision = (
            "EXERCISE"
            if excercise > continuation
            else "HOLD"
        )

        level_info.append([
            n,
            j,
            round(S,4),
            round(continuation,4),
            round(excercise,4),
            decision
        ])    
        new_vals.append(value)

    decision_tables.extend(level_info)
    american=new_vals

decision_df = pd.DataFrame(
    decision_tables,
    columns=[
        "time",
        "node",
        "stock",
        "continuation",
        "exercise",
        "decision"
    ]
)
print(decision_df)
   

   time  node     stock  continuation  exercise  decision
0     2     0  133.4658        0.0000    0.0000      HOLD
1     2     1  100.0000        7.0313    0.0000      HOLD
2     2     2   74.9256       24.5289   25.0744  EXERCISE
3     1     0  115.5274        3.6784    0.0000      HOLD
4     1     1   86.5596       16.3313   13.4404      HOLD
5     0     0  100.0000       10.2249    0.0000      HOLD


In [33]:
american_price = american[0]
european_price = euro[-1][0]
##Python allows negative indexing so euro[-1][0] actually means euro[euro.size()-1][0]
premium = american_price - european_price

print("European:", round(european_price,4))
print("American:", round(american_price,4))
print("Premium :", round(premium,4))

European: 7.762
American: 10.2249
Premium : 2.4629


In [34]:
boundary = decision_df[
    decision_df["decision"] == "EXERCISE"
]

print(boundary)

   time  node    stock  continuation  exercise  decision
2     2     2  74.9256       24.5289   25.0744  EXERCISE


In [35]:
fu = euro[-2][0]
fd = euro[-2][1]
delta = (
    (fu - fd)
    / (S0*u - S0*d)
)
print("Delta =", round(delta,4))

Delta = -0.3784


REFLECTION:
American put options command a higher premium than their European counterparts because they provide the holder with the additional flexibility of exercising the option before maturity. In this module, I learned the foundations of the Binomial Asset Pricing Model and its derivation through the construction of replicating portfolios with identical payoffs. By equating the payoffs of these portfolios, I gained an understanding of the no-arbitrage principle and its role in option valuation.
I also studied the concept of risk-neutral probabilities, which significantly simplify the pricing process by allowing expected option values to be calculated under a risk-neutral measure. Furthermore, I learned the derivation of the Cox–Ross–Rubinstein (CRR) parameters, including the up factor, down factor, and risk-neutral probability used in the binomial tree framework.
As part of the assignment, I implemented a three-step binomial tree model to price an American put option and identify its early-exercise boundary. Through this exercise, I observed that the optimal exercise decision occurred only when the put option was sufficiently deep in the money. In such regions, the immediate exercise value exceeded the continuation value, making early exercise economically rational. This observation aligns with the theoretical intuition that the flexibility embedded in American put options becomes most valuable when the option is deep in the money and interest rates are positive.

